# KNearestNeighbours с логарифмом цены

In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder
import pickle

```
Загрузка датасета
```

In [3]:
df = pd.read_csv('data/drom_archive_cleaned_2018-2025.csv')

```
Создание колонки с логарифмом цены
```

In [4]:
df['log_price'] = np.log(df['Цена'])

```
Преобразование категориальных переменных
```

In [5]:
categorical = ['Название машины', 'Тип двигателя', 'Коробка передач', 'Привод', 'Руль', 'Поколение', 'Рестайлинг',
               'Тип кузова', 'Метка', 'Город']
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Год объявления', 'Месяц объявления', 'Возраст авто']

In [6]:
for col in categorical:
    le = LabelEncoder() # используем LabelEncoder, поскольку он больше подходит для knn
    df[col] = le.fit_transform(df[col])

```
Разделение данных на целевую переменную и признаки
```

In [7]:
y = df['log_price']
X = df.drop(['Цена', 'log_price'], axis=1)

```
Разделение на обучающую и тестовую выборки (2018-2023, 2024-2025)
```

In [8]:
X_train = X[X['Год объявления'] <= 2023]
X_test = X[X['Год объявления'] >= 2024]

y_train = y[X['Год объявления'] <= 2023]
y_test = y[X['Год объявления'] >= 2024]
joblib.dump((X_train, X_test, y_train, y_test), "data/data_split_label_log.pkl")
# Сохранение выборки с LabelEncoder
X_train, X_test, y_train, y_test = joblib.load("data/data_split_label_log.pkl")

In [9]:
model = KNeighborsRegressor(n_neighbors=5, n_jobs=-1)

```
Обучение и сохранение модели
```

In [10]:
model.fit(X_train, y_train)
joblib.dump(model, "models/knn_log_model.pkl")

['models/knn_log_model.pkl']

```
Предсказание на тестовой выборке
```

In [11]:
y_pred_log = model.predict(X_test)

In [12]:
y_pred = np.exp(y_pred_log) # Обратное преобразование прогнозов
y_test_real = np.exp(y_test)

```
Оценка модели
```

In [13]:
knn_mse = mean_squared_error(y_test_real, y_pred)
knn_rmse = np.sqrt(knn_mse)
knn_mae = mean_absolute_error(y_test_real, y_pred)
knn_r2 = r2_score(y_test_real, y_pred)

```
Вывод результатов
```

In [14]:
pd.options.display.float_format = '{:_.2f}'.format
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)', 'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)'],
    'Результаты': [knn_mse, knn_rmse, knn_mae, knn_r2]
})

,Метод оценки,Результаты
0,Среднеквадратическая ошибка (MSE),872_067_577_121.76
1,Среднеквадратическая ошибка (RMSE),933_845.59
2,Средняя абсолютная ошибка (MAE),538_566.21
3,Коэффицент детерминации (R^2),0.17


```
Сохранение метрик в отдельный файл
```

In [15]:
metrics = {
    "model_name": "KNearestNeighbors Log",
    "MSE": knn_mse,
    "RMSE": knn_rmse,
    "MAE": knn_mae,
    "R2": knn_r2
}

In [16]:
with open("metrics/knn_log_metrics.pkl", "wb") as f:
    pickle.dump(metrics, f)